# Treino DQN, interseccao Eduardo Mondlane / Salvador Allende (Google Colab)

Corre o treino completo do agente DQN sem ocupar a maquina local. Ver `PROGRESSO.md` no repositorio para o contexto completo.

**Como usar:**
1. Corre as celulas por ordem.
2. Na celula de clone, cola um *personal access token* do GitHub quando pedido (scope `repo`, porque o repositorio e privado). O token nao fica gravado no notebook.
3. Escolhe `CENARIO` e `SEMENTE` na celula de configuracao. Para treinar as 5 sementes em paralelo, abre 5 sessoes Colab (uma por semente) com este mesmo notebook.
4. No fim, a ultima celula faz commit e push do CSV de resultados para o repositorio, para juntares tudo depois na tua maquina.

## 1. Instalar o SUMO

In [1]:
!apt-get update -qq
!apt-get install -y -qq sumo sumo-tools sumo-doc
import os
os.environ["SUMO_HOME"] = "/usr/share/sumo"
!sumo --version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package binfmt-support.
(Reading database ... 122797 files and directories currently installed.)
Preparing to unpack .../00-binfmt-support_2.2.2-7_amd64.deb ...
Unpacking binfmt-support (2.2.2-7) ...
Selecting previously unselected package fonts-roboto-unhinted.
Preparing to unpack .../01-fonts-roboto-unhinted_2%3a0~20170802-3_all.deb ...
Unpacking fonts-roboto-unhinted (2:0~20170802-3) ...
Selecting previously unselected package fastjar.
Preparing to unpack .../02-fastjar_2%3a0.98-7_amd64.deb ...
Unpacking fastjar (2:0.98-7) ...
Selecting previously unselected package jarwrapper.
Preparing to unpack .../03-jarwrapper_0.79_all.deb ...
Unpacking jarwrapper (0.79) ...
Selecting previously unselected package javascript-common.
Preparing to unpack .../04-javascript-common_

## 2. Clonar o repositorio privado

Precisas de um *personal access token* do GitHub (Settings > Developer settings > Fine-grained tokens, scope de leitura/escrita sobre este repositorio, ou um classic token com scope `repo`). Cola-o quando pedido; fica so na memoria desta sessao Colab.

In [2]:
from getpass import getpass

token = getpass("Cola aqui o teu GitHub personal access token: ")
utilizador = "Shads-GoldenCorsair"
repo = "sumo-interseccao-eduardo-mondlane-salvador-allende"

!git clone https://{token}@github.com/{utilizador}/{repo}.git
%cd {repo}
del token  # nao deixa o token na variavel apos clonar

Cola aqui o teu GitHub personal access token: ··········
Cloning into 'sumo-interseccao-eduardo-mondlane-salvador-allende'...
remote: Enumerating objects: 325, done.
remote: Counting objects: 100% (325/325), done.
remote: Compressing objects: 100% (204/204), done.
remote: Total 325 (delta 206), reused 232 (delta 118), pack-reused 0 (from 0)
Receiving objects: 100% (325/325), 3.63 MiB | 7.34 MiB/s, done.
Resolving deltas: 100% (206/206), done.
/content/sumo-interseccao-eduardo-mondlane-salvador-allende


## 3. Instalar dependencias Python

In [3]:
!pip install -q -r agente_dqn/requirements.txt

## 4. Configuracao desta sessao

Muda `CENARIO` e `SEMENTE` conforme a sessao. Cada sessao Colab so treina **uma** semente (para poderes correr varias sessoes em paralelo, uma por semente).

In [4]:
# Célula 4
CENARIO = "baixo_fluxo"  # ou "pico"
SEMENTES = 5
EPISODIOS = 100


## 4b. Ligar checkpoints ao Google Drive

O `train.py` grava progresso (pesos + episodio actual) a cada 10 episodios, para retomar se a sessao for interrompida. Mas o disco do Colab e efemero: se a sessao for reiniciada (nao so desligada por inactividade), o que la estiver perde-se. Esta celula guarda os checkpoints no teu Google Drive em vez disso. Vai pedir autorizacao de acesso ao Drive na primeira vez.

In [5]:
from google.colab import drive
drive.mount("/content/drive")

import os
pasta_drive = f"/content/drive/MyDrive/pfc_checkpoints_dqn/{CENARIO}"
os.makedirs(pasta_drive, exist_ok=True)

!rm -rf outputs/_checkpoints
!ln -s "{pasta_drive}" outputs/_checkpoints
print("checkpoints desta sessao gravados em:", pasta_drive)

Mounted at /content/drive
checkpoints desta sessao gravados em: /content/drive/MyDrive/pfc_checkpoints_dqn/baixo_fluxo


## 5. Treinar

Fica a correr nesta celula. O Colab gratuito desliga por inactividade (~90min sem interaccao) ou ao fim de algumas horas de sessao continua. Se isso acontecer, volta a abrir o notebook e corre as celulas por ordem outra vez (incluindo a 4b): o `train.py` deteta o checkpoint no Drive e retoma a partir do ultimo episodio gravado, em vez de recomecar do zero.

In [ ]:
# Célula 5
!cd agente_dqn && python -u train.py --cenario {CENARIO} --episodios {EPISODIOS} --sementes {SEMENTES}

2026-09-16 09:02:00.650040: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-16 09:02:00.769329: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-09-16 09:02:07.530515: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
 Retrying in 1 seconds
***Starting server on port 37691 ***
Loading net-file from '/content/sumo-interseccao-eduardo-mondlane-salvador-allende/agente_dqn/../config/../net/eduardo_mondlane_salvador_allende.net.xml' ... done (4ms).
Loading additional-files from '/content/sumo-interseccao-eduardo-mondlane-salvador-allende/agente_dqn/../config/../net/paragens

## 6. Enviar os resultados de volta para o GitHub

Grava so o CSV de resultados desta semente (nao os pesos da rede, para nao complicar); junta os CSVs das 5 sementes na tua maquina depois.

In [ ]:
# Célula 6
!git config user.email "colab@treino.local"
!git config user.name "Treino Colab"
!git add outputs/treino_{CENARIO}.csv
!git commit -m "Resultados treino {CENARIO}, 5 sementes"
!git pull --no-edit --no-rebase origin master
!git push origin master


In [ ]:
!pwd
!ls /content/
